# 03 — Direct Preference Optimization (DPO)

## What DPO means and when to use it

**Direct Preference Optimization (DPO)** learns from comparisons: for the same prompt, one response is preferred to another. A **preference pair** records a prompt plus chosen and rejected responses.

The **policy** is the model being trained to generate responses. A **reference policy** is a frozen comparison model, normally related to the initial policy. **Frozen** means its weights are not updated. The reference helps express preference changes relative to a starting behavior.

**What problem does it solve?** DPO is useful when it is easier to rank candidate responses than write one ideal answer. It uses supplied pairs directly, without first training a separate reward model in this workflow. Incorrect, inconsistent or length-biased rankings can still teach undesirable preferences.

**Supervised Fine-Tuning (SFT)** imitates demonstrations. DPO additionally uses a rejected alternative to express a relative preference.


## NovaBot preference pair

**Fictional explanatory record:**

~~~json
{"prompt":"How many modes does NovaBot have?",
 "chosen":"Three: idle, mapping, navigation.",
 "rejected":"Five modes."}
~~~

The desired change is to favor the chosen answer relative to the rejected one, compared with the reference policy's preference. The label does not prove that every other possible answer is bad.

A **log-probability** is the natural logarithm of a probability. Token probabilities multiply along a response; their log-probabilities add. The notebook sums over answer tokens only, with both responses conditioned on the same prompt. Dividing by response length would change the displayed standard sigmoid-DPO objective.


## How the objective works

Let $x$ be the prompt, $y^+$ the chosen response, $y^-$ the rejected response, $\pi_\theta$ the trainable policy, and $\pi_{\mathrm{ref}}$ the frozen reference. Define

$$
m_\theta=\log\pi_\theta(y^+\mid x)-\log\pi_\theta(y^-\mid x),\quad
m_{\mathrm{ref}}=\log\pi_{\mathrm{ref}}(y^+\mid x)-\log\pi_{\mathrm{ref}}(y^-\mid x),
$$
$$z=\beta(m_\theta-m_{\mathrm{ref}}),\qquad L=-\log\sigma(z).$$

$m_\theta$ and $m_{\mathrm{ref}}$ are **preference margins**; $\beta>0$ scales the relative margin; $\sigma(z)=1/(1+\exp(-z))$ is the sigmoid function; $\exp$ is the exponential. The batch loss averages this value over pairs. A larger positive $z$ lowers the loss. Beta's practical effect depends on the training setup; it is not a simple “quality” knob.

**Hand calculation:** policy log-probabilities -2 and -3 give margin 1; reference values -3 and -3 give margin 0. With beta 0.1, $z=0.1$ and loss is about 0.644, compared with about 0.693 at $z=0$. Those are constructed numbers, not observed model outputs.


## Connection to this notebook

preference_batches() constructs chosen and rejected sequences. sequence_logprob() scores only answer targets using a **loss mask**, a selection of scored token positions. dpo_objective() subtracts reference margins inside no_grad(), so no gradients flow into the reference.

The cell checks that negative log-sigmoid matches binary cross-entropy with target one. The heldout_margin reported here is the reference-adjusted, beta-scaled $z$, not simply the policy's raw $m_\theta$. Its accuracy metric counts pairs with $z>0$; it is not factual question-answer accuracy.

The policy uses **Low-Rank Adaptation (LoRA)**, small trainable matrix additions; the local quantized profile uses **Quantized Low-Rank Adaptation (QLoRA)**. The actual pairs are the existing instructional data, with the same structure as NovaBot's pair. A separate one-step Trainer comparison links the visible formula to the framework.


## Common confusions and quick check

A lower preference loss does not prove that individual facts are correct. DPO changes relative preference and does not require a separately trained scalar reward model in this setup.

1. If policy and reference margins are equal, what is the sigmoid-DPO loss?
2. Should the reference receive gradients when computing a pair's loss?

<details>
<summary>Answers</summary>

1. $z=0$, so the loss is $-\log(0.5)\approx0.693$, regardless of the common raw margin.
2. No. Its frozen scores provide the comparison anchor. The policy, not the reference, is updated.

</details>


## Before running the experiment

**Learning goals:** derive answer log-probabilities, compare policy and frozen reference margins, calculate DPO loss and update an adapter.

Run top to bottom in a fresh kernel. The tiny profile uses real Qwen classes with random weights. These are mechanics experiments, not quality benchmarks. [Course index](README.md) · [Flow diagrams](../docs/EXECUTION_AND_DATA_FLOW.md)

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Local experiment parameters

For local weights, set `FTLAB_DEVICE` to `cpu`, `mps`, `cuda`, or `auto` before launching. CPU/MPS default to ordinary LoRA; CUDA retains its QLoRA profile. Restart the kernel when changing devices after a Trainer has initialized. Selecting a device does not guarantee the full experiment fits its memory.


In [ ]:
LESSON = "03"
# Parameters: change these before running the notebook from top to bottom.
import csv
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.devices import (
    activate_runtime,
    empty_device_cache,
    resolve_runtime,
)
from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
# tiny_cpu remains an offline CPU fixture; real checkpoints use the selected backend.
REQUESTED_DEVICE = "cpu" if MODE == "tiny_cpu" else os.environ.get("FTLAB_DEVICE", "auto")
RUNTIME = resolve_runtime(device=REQUESTED_DEVICE, dtype=os.environ.get("FTLAB_DTYPE"))
activate_runtime(RUNTIME)
DEVICE = torch.device(RUNTIME.device)
DTYPE = RUNTIME.torch_dtype
ATTENTION = "eager" if MODE == "tiny_cpu" else RUNTIME.attention
print(RUNTIME.report())
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

## Load weights and preprocessing

The checkpoint fixes the tokenizer and chat template. A reference or teacher must use a compatible vocabulary. It is frozen but still consumes memory.


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "device_map": {"": str(DEVICE)},
    "attn_implementation": ATTENTION,
}
# Quantization is a separate choice. CPU/MPS use unquantized LoRA.
_default_qlora = (
    MODE == "local_pretrained" and DEVICE.type == "cuda" and LESSON not in {"00a", "00b", "04"}
)
USE_QLORA = os.environ.get("FTLAB_USE_QLORA", str(_default_qlora)).lower() in {"1", "true", "yes"}
if USE_QLORA and DEVICE.type != "cuda":
    raise ValueError("This CPU/MPS profile supports ordinary LoRA; set FTLAB_USE_QLORA=false.")
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
import re


def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=int(os.environ.get("FTLAB_LORA_RANK", "4" if MODE == "tiny_cpu" else "16")),
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
if MODE == "local_pretrained" and not USE_QLORA:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Establish a baseline

Use `eval()` plus `no_grad()` for measurement, and switch back to `train()` for updates. We aggregate causal loss by supervised token count. Greedy decoding makes the before/after and reload comparisons reproducible on the same device.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return RUNTIME.precision_context()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt="What is the color of sky ?"):
    current_model.eval()
    text = render([{"role": "user", "content": prompt}], generation=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    inputs.pop("token_type_ids", None)
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_validation_loss": baseline_loss,
        "baseline_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## Construct preferred and rejected answer batches

Both responses see the identical prompt. Sum log-probabilities over answer tokens only; averaging by response length changes the standard sigmoid DPO objective. The frozen reference anchors the preference update to the initial policy.


In [ ]:
def sequence_logprob(current_model, encoded):
    # Sum only answer-token log-probabilities. Prompts condition both candidates.
    logits = (
        current_model(**{k: v for k, v in encoded.items() if k != "labels"}, use_cache=False)
        .logits[:, :-1]
        .float()
    )
    labels = encoded["labels"][:, 1:]
    valid = labels != -100
    targets = labels.masked_fill(~valid, 0)
    token_logp = logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return (token_logp * valid).sum(-1)


def preference_batches(rows):
    if any(not r["rejected"].strip() or r["chosen"] == r["rejected"] for r in rows):
        raise ValueError("Preferences need a nonempty rejected answer distinct from chosen.")
    chosen = [
        {
            "messages": [
                {"role": "user", "content": row["prompt"]},
                {"role": "assistant", "content": row["chosen"]},
            ]
        }
        for row in rows
    ]
    rejected = [
        {
            "messages": [
                {"role": "user", "content": row["prompt"]},
                {"role": "assistant", "content": row["rejected"]},
            ]
        }
        for row in rows
    ]
    return to_device(collate_text(chosen)), to_device(collate_text(rejected))


reference = (
    AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        dtype=DTYPE,
        attn_implementation=ATTENTION,
    )
    .to(DEVICE)
    .eval()
)
reference.requires_grad_(False)
chosen_batch, rejected_batch = preference_batches(splits["train"][:2])
BETA = 0.1


def dpo_objective(current_model, rows):
    chosen, rejected = preference_batches(rows)
    policy_margin = sequence_logprob(current_model, chosen) - sequence_logprob(
        current_model, rejected
    )
    with torch.no_grad():
        reference_margin = sequence_logprob(reference, chosen) - sequence_logprob(
            reference, rejected
        )
    preference_logits = BETA * (policy_margin - reference_margin)
    return -F.logsigmoid(preference_logits).mean(), preference_logits


model.eval()
with torch.no_grad():
    preference_before, _ = dpo_objective(model, splits["validation"])
print("Initial DPO loss:", float(preference_before))

## One explicit preference update

For sigmoid DPO, `-logsigmoid(beta * (policy_margin - reference_margin))` is also binary cross-entropy with target one. This identity provides an independent numerical check. Reference parameters must never receive gradients.


In [ ]:
model.train()
optimizer = torch.optim.AdamW(trainable, lr=2e-3 if MODE == "tiny_cpu" else 2e-4)
optimizer.zero_grad(set_to_none=True)
with precision_context():
    loss, preference_logits = dpo_objective(model, splits["train"][:2])
    expected = F.binary_cross_entropy_with_logits(
        preference_logits, torch.ones_like(preference_logits)
    )
torch.testing.assert_close(loss, expected)
assert torch.isfinite(loss)
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
assert all(p.grad is None for p in reference.parameters())
torch.nn.utils.clip_grad_norm_(trainable, 1.0)
optimizer.step()
model.eval()
with torch.no_grad():
    preference_after, heldout_logits = dpo_objective(model, splits["validation"])
print(
    {
        "dpo_loss": float(loss.detach()),
        "heldout_dpo_loss": float(preference_after),
        "heldout_preference_accuracy": float((heldout_logits > 0).float().mean()),
        "heldout_margin": float(heldout_logits.mean()),
    }
)

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Release training models, optimizer state and graph references before independent reload.
for _name in (
    "model",
    "optimizer",
    "scheduler",
    "trainable",
    "parameter",
    "p",
    "outputs",
    "loss",
    "reference",
    "teacher",
    "reward_model",
    "value_model",
    "policy_optimizer",
    "value_optimizer",
    "reward_optimizer",
    "weights",
    "state",
    "logits",
    "student_logits",
    "teacher_logits",
    "new_token_logp",
    "new_logp",
    "ratio",
    "policy_loss",
    "value_loss",
    "reward_loss",
    "margin",
    "per_token_divergence",
):
    globals().pop(_name, None)
empty_device_cache(DEVICE)

# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "runtime": RUNTIME.report(),
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
empty_device_cache(DEVICE)

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.


## Connect this calculation to FineTuneLab

The recipe builds the production trainer from the same canonical records. The CPU profile executes a separate one-step comparison. The real multi-model comparison is shown explicitly but is deferred to a fresh process to avoid keeping two experiments in GPU memory.


In [ ]:
METHOD = "dpo"
framework_rows = [
    {"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]}
    for r in splits["train"][:2]
]
# Map the visible experiment back to the framework boundary.
# Start a separate short run from the same base, not from the mutated notebook model.
from datasets import Dataset

from finetunelab.config import RECIPE_ADAPTER
from finetunelab.data import validate_dataset
from finetunelab.recipes import build_trainer

recipe = {
    "method": METHOD,
    "model": {
        "name_or_path": str(MODEL_PATH),
        "local_files_only": True,
        "dtype": RUNTIME.dtype,
    },
    "data": {"source": str(OUTPUT / "train.jsonl"), "max_length": 128},
    "tuning": {"strategy": "lora", "lora_rank": 4, "lora_alpha": 8},
    "training": {
        "output_dir": str(OUTPUT / "framework"),
        "max_steps": 1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "gradient_checkpointing": False,
        "device": RUNTIME.device,
        "bf16": RUNTIME.bf16,
        "fp16": RUNTIME.fp16,
        "tf32": False,
        "logging_steps": 1,
        "save_steps": 1,
        "report_to": ["none"],
    },
    "generation": {"max_new_tokens": 4},
}
config = RECIPE_ADAPTER.validate_python(recipe)
framework_dataset = Dataset.from_list(framework_rows)
print(validate_dataset(framework_dataset, config))
# True in CPU acceptance tests. Real multi-model runs need sufficient GPU memory.
RUN_FRAMEWORK_COMPARISON = MODE == "tiny_cpu"
if RUN_FRAMEWORK_COMPARISON:
    trainer = build_trainer(config, framework_dataset)
    framework_result = trainer.train()
    trainer.save_model(str(OUTPUT / "framework" / "final"))
    print(framework_result.metrics)
    assert all(
        math.isfinite(float(v))
        for v in framework_result.metrics.values()
        if isinstance(v, (int, float))
    )
    del trainer
else:
    print(
        "Configuration validated. In a fresh process run "
        "build_trainer(config, framework_dataset).train()."
    )